In [1]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Sweet Crumbs Orders")
ws = sh.sheet1

ws.append_row([
    "name", "item", "qty", "date", "eggless", "total"
])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1kSH2bd9tKK7-tWZZN0nL0Jp1tGqT9Ht1UEOXUNoMNVg


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""

    name: str = Field(description="Customer's first name")

    item: Literal[
        "plum cake",
        "chocolate cake",
        "cupcake"
    ] = Field(description="Menu item")

    qty: int = Field(description="How many", ge=1)

    date: str = Field(
        description="Pickup date as the customer said it"
    )

    eggless: bool = Field(
        description="True if the customer asked for eggless"
    )

In [3]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())

    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."

    return f"{qty} x {item} = Rs {price * qty}"


@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week."""

    hours = HOURS.get(day.lower().strip())

    return (
        f"Open {hours} on {day}."
        if hours
        else f"'{day}' is not a day of the week."
    )


@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name,
    item, quantity and pickup date.
    """

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return (
        f"Saved: {order.qty} x {order.item} for "
        f"{order.name} on {order.date}, Rs {total}."
    )

In [4]:
from langchain.agents import create_agent

SYSTEM = """ROLE: You are the friendly order assistant for Sweet Crumbs Home Bakery.

FACTS: Orders need 24 hours' notice. All cakes have eggless versions.

RULES:
Always use get_price for prices and is_open for timings; never guess them.
Before log_order, make sure you have name, item, quantity and pickup date.
Ask the customer if any required detail is missing.
After saving, read the order back with the total.

FORMAT:
Under 60 words, friendly, English or Telugu matching the customer.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

history = []
last_result = None

def chat(text: str) -> str:
    global history, last_result

    try:
        last_result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = last_result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

In [6]:
print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))

Yes, we’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost ₹1,300. If you’d like the eggless version, just let me know your name and pickup date and I’ll log the order for you.


In [7]:
print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))

Great! Your order is confirmed: 2 eggless chocolate cakes for Ravi on 24 Dec. Total ₹1,300. See you then!


In [8]:
@tool
def check_order_availability(hours_until_order: int) -> str:
    """Check whether an order meets the 24-hour advance notice policy."""

    if hours_until_order >= 24:
        return "Order can be accepted."

    return "Order requires at least 24 hours' notice."

In [9]:
print(check_order_availability.invoke({
    "hours_until_order": 48
}))

print(check_order_availability.invoke({
    "hours_until_order": 12
}))

Order can be accepted.
Order requires at least 24 hours' notice.


In [10]:
extractor = llm.with_structured_output(Order)

messages = [
    "hi ravi here, need 2 plum cakes for 24th dec, eggless pls",
    "I am Sita. Can I get one chocolate cake on Sunday?",
    "Hey, John wants 5 cupcakes for 28 December, not eggless",
    "This is Ahmed, please arrange 3 chocolate cakes for tomorrow, eggless",
    "Hello, I'm Priya. Need 4 cupcakes on 30 Dec."
]

for message in messages:
    try:
        order = extractor.invoke(message)
        print("\nCustomer message:", message)
        print("Structured order:", order)
    except Exception as e:
        print("\nCould not extract order:", e)


Customer message: hi ravi here, need 2 plum cakes for 24th dec, eggless pls
Structured order: name='Ravi' item='plum cake' qty=2 date='24th Dec' eggless=True

Customer message: I am Sita. Can I get one chocolate cake on Sunday?
Structured order: name='Sita' item='chocolate cake' qty=1 date='Sunday' eggless=False

Customer message: Hey, John wants 5 cupcakes for 28 December, not eggless
Structured order: name='John' item='cupcake' qty=5 date='28 December' eggless=False

Customer message: This is Ahmed, please arrange 3 chocolate cakes for tomorrow, eggless
Structured order: name='Ahmed' item='chocolate cake' qty=3 date='tomorrow' eggless=True

Customer message: Hello, I'm Priya. Need 4 cupcakes on 30 Dec.
Structured order: name='Priya' item='cupcake' qty=4 date='30 Dec' eggless=False


In [12]:
invalid_messages = [
    "I want 2 donuts.",
    "I need 0 cupcakes."
]

for message in invalid_messages:
    try:
        order = extractor.invoke(message)
        print(order)
    except Exception as e:
        print("Invalid order:", e)

Invalid order: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'I’m sorry, but we don’t have donuts on our menu. We do offer plum cake, chocolate cake, and cupcakes. Would you like to order one of those instead?'}}
Invalid order: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'I’m sorry, but we can’t process an order for 0 cupcakes. Could you let me know how many you’d like to order?'}}


In [13]:
invalid_messages = [
    "I want 2 donuts.",
    "I need 0 cupcakes."
]

for message in invalid_messages:
    try:
        order = extractor.invoke(message)
        print(order)
    except Exception as e:
        print("Invalid order:", e)

Invalid order: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'I’m sorry, but we don’t have donuts on the menu. We do offer plum cake, chocolate cake, and cupcakes. Would you like to place an order for one of those instead?'}}
Invalid order: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'I’m sorry, but we can’t process an order for zero cupcakes. If you’d like to place an order for at least one, just let me know the details!'}}


In [15]:
@tool("tool1")
def get_price(item: str, qty: int = 1) -> str:
    """does stuff"""
    price = MENU.get(item.lower().strip())

    if price is None:
        return "Item not available."

    return f"Total price: Rs {price * qty}"

In [16]:
print(chat("How much are 2 chocolate cakes?"))

Two chocolate cakes cost ₹1,300.


In [18]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the price of 2 chocolate cakes?"}
    ]
})

print(result["messages"][-1].content)

Sure! Two chocolate cakes will cost you Rs 1300. Let me know if you’d like to place an order or need anything else!


In [19]:
trace_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Are you open on Sunday and what is the price of 2 cupcakes?"
        }
    ]
})

for message in trace_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Are you open on Sunday and what is the price of 2 cupcakes?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_546ac48c-5092-4759-8e25-13f9c6b6ba15)
 Call ID: fc_546ac48c-5092-4759-8e25-13f9c6b6ba15
  Args:
    day: Sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on Sunday.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_c8326a69-9f19-4ae1-9cb7-0bbf944aa9ad)
 Call ID: fc_c8326a69-9f19-4ae1-9cb7-0bbf944aa9ad
  Args:
    item: cupcake
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x cupcake = Rs 160
================================== Ai Message ==================================

Yes! We’re open Sunday from 10 am to 6 pm. Two cupcakes cost ₹160. Let me know

In [20]:
from langchain.messages import AIMessage, ToolMessage

messages = trace_result["messages"]

model_messages = [
    m for m in messages
    if isinstance(m, AIMessage)
]

tool_messages = [
    m for m in messages
    if isinstance(m, ToolMessage)
]

print("AI messages:", len(model_messages))
print("Tool results:", len(tool_messages))

AI messages: 3
Tool results: 2
